# Retraining the bird species classifier

The original classifier was a small CNN trained on **64x64** crops, which is a lot of
detail to throw away - in the project report it calls an Indian Peacock at 47% and a
Cattle Egret at 23%.

This notebook fine-tunes a pretrained backbone at **224x224** instead, on the same 25
species, and exports both a PyTorch checkpoint and an ONNX file (the web app runs the
ONNX one in the browser).

Runtime -> Change runtime type -> **GPU** before running.

Dataset: [birds25-cleaned](https://www.kaggle.com/datasets/pavangawande/birds25-cleaned)
by pavangawande, CC BY-NC 4.0 - non-commercial use only, which is what this is.

## 1. Kaggle credentials

Kaggle -> your profile -> Settings -> API -> *Create New Token*. That downloads
`kaggle.json`. Run the cell and upload it - it stays in this Colab session only.

In [ ]:
from google.colab import files
import os, json, pathlib

if not pathlib.Path('/root/.kaggle/kaggle.json').exists():
    print('Upload your kaggle.json:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(next(iter(uploaded.values())))
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('credentials ready')

## 2. Get the dataset

~16 GB, so this takes a few minutes. Colab's disk is temporary, which is exactly why
we train here instead of on a laptop.

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d pavangawande/birds25-cleaned -p /content --unzip
!du -sh /content/* | head

In [ ]:
import pathlib

# find the folder that actually holds the per-species subfolders
def find_image_root(start='/content'):
    best, best_count = None, 0
    for path in pathlib.Path(start).rglob('*'):
        if not path.is_dir():
            continue
        subdirs = [d for d in path.iterdir() if d.is_dir()]
        if len(subdirs) < 10:
            continue
        images = sum(1 for d in subdirs for _ in d.glob('*.jpg'))
        if images > best_count:
            best, best_count = path, images
    return best, best_count

IMAGE_ROOT, total = find_image_root()
print('images root:', IMAGE_ROOT, '| images:', total)
for d in sorted(IMAGE_ROOT.iterdir())[:30]:
    if d.is_dir():
        print(f'  {d.name:34} {len(list(d.glob("*.jpg")))}')

## 3. Match the app's 25 species

The app can only name the species it has sounds for, so we train on exactly those and
in the same order - the ONNX output index has to line up with the app's class list.

In [ ]:
CLASS_NAMES = [
    'Asian Green Bee Eater', 'Brown Headed Barbet', 'Cattle Egret',
    'Common Kingfisher', 'Common Myna', 'Common Rosefinch',
    'Common Tailorbird', 'Coppersmith Barbet', 'Forest Wagtail',
    'Gray Wagtail', 'Hoopoe', 'House Crow',
    'Indian Grey Hornbill', 'Indian Peacock', 'Indian Pitta',
    'Indian Roller', 'Jungle Babbler', 'Northern Lapwing',
    'Red Wattled Lapwing', 'Ruddy Shelduck', 'Rufous Treepie',
    'Sarus Crane', 'White Breasted Kingfisher',
    'White Breasted Waterhen', 'White Wagtail',
]

def normalise(name):
    return ''.join(ch for ch in name.lower() if ch.isalnum())

folders = {normalise(d.name): d for d in IMAGE_ROOT.iterdir() if d.is_dir()}

matched, missing = {}, []
for species in CLASS_NAMES:
    folder = folders.get(normalise(species))
    if folder is None:  # try a looser match
        candidates = [f for key, f in folders.items() if normalise(species) in key or key in normalise(species)]
        folder = candidates[0] if len(candidates) == 1 else None
    if folder is None:
        missing.append(species)
    else:
        matched[species] = folder

print(f'matched {len(matched)}/{len(CLASS_NAMES)} species')
if missing:
    print('NOT FOUND - check these against the folder list above:')
    for m in missing: print('  ', m)

## 4. Build train/val splits

A fixed seed and a per-species split, so validation numbers are comparable between runs.

In [ ]:
import random, shutil, os

random.seed(1337)
WORK = pathlib.Path('/content/split')
VAL_FRACTION = 0.15

if WORK.exists(): shutil.rmtree(WORK)
counts = {}
for species, folder in matched.items():
    images = sorted(p for p in folder.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    random.shuffle(images)
    cut = max(1, int(len(images) * VAL_FRACTION))
    for split, chunk in (('val', images[:cut]), ('train', images[cut:])):
        target = WORK / split / species
        target.mkdir(parents=True, exist_ok=True)
        for src in chunk:
            os.symlink(src, target / src.name)   # symlink: no copying 16 GB around
    counts[species] = (len(images) - cut, cut)

print(f"{'species':34}{'train':>7}{'val':>6}")
for species, (tr, va) in counts.items(): print(f'{species:34}{tr:>7}{va:>6}')
print(f"\ntotal train {sum(t for t,_ in counts.values())}, val {sum(v for _,v in counts.values())}")

## 5. Fine-tune

EfficientNet-B0, pretrained on ImageNet. The backbone already knows edges, feathers and
textures; we mostly teach it the 25 labels. Far better than learning from scratch on a
few thousand photos, which is what the original CNN had to do.

In [ ]:
import torch, torch.nn as nn, time
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE, BATCH, EPOCHS = 224, 64, 8
print('device:', DEVICE)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(WORK / 'train', train_tf)
val_ds = datasets.ImageFolder(WORK / 'val', val_tf)

# ImageFolder sorts classes alphabetically; make sure that ordering is the one we ship
ORDERED_CLASSES = train_ds.classes
print('classes:', len(ORDERED_CLASSES))

train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, BATCH, shuffle=False, num_workers=2, pin_memory=True)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(ORDERED_CLASSES))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, 3e-4, epochs=EPOCHS, steps_per_epoch=len(train_dl))

def evaluate():
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            pred = model(x.to(DEVICE)).argmax(1).cpu()
            correct += (pred == y).sum().item(); total += y.size(0)
    return correct / total

best = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train(); started = time.time(); running = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step(); scheduler.step()
        running += loss.item() * x.size(0)
    accuracy = evaluate()
    print(f'epoch {epoch}/{EPOCHS}  loss {running/len(train_ds):.3f}  val acc {accuracy:.1%}  ({time.time()-started:.0f}s)')
    if accuracy > best:
        best = accuracy
        torch.save(model.state_dict(), '/content/bird_classifier.pth')
print(f'\nbest validation accuracy: {best:.1%}')

## 6. Where it still gets confused

Per-species accuracy is more honest than one headline number - it shows which birds the
model actually struggles with (the two kingfishers and the three wagtails, usually).

In [ ]:
from collections import defaultdict

model.load_state_dict(torch.load('/content/bird_classifier.pth'))
model.eval()

right, seen = defaultdict(int), defaultdict(int)
confused = defaultdict(int)
with torch.no_grad():
    for x, y in val_dl:
        preds = model(x.to(DEVICE)).argmax(1).cpu()
        for true, pred in zip(y.tolist(), preds.tolist()):
            seen[true] += 1
            if true == pred: right[true] += 1
            else: confused[(ORDERED_CLASSES[true], ORDERED_CLASSES[pred])] += 1

print(f"{'species':34}{'accuracy':>10}{'n':>6}")
for i, name in enumerate(ORDERED_CLASSES):
    if seen[i]: print(f'{name:34}{right[i]/seen[i]:>9.0%}{seen[i]:>6}')

print('\nmost common mix-ups:')
for (true, pred), n in sorted(confused.items(), key=lambda kv: -kv[1])[:10]:
    print(f'  {n:>3}x  {true}  ->  {pred}')

## 7. Export for the app

ONNX for the browser, the checkpoint for the desktop version, and a small JSON so the
app knows the class order and preprocessing without guessing.

In [ ]:
import json

model.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

torch.onnx.export(
    model, dummy, '/content/bird_classifier.onnx',
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)

meta = {
    'classes': ORDERED_CLASSES,
    'input_size': IMG_SIZE,
    'mean': MEAN,
    'std': STD,
    'architecture': 'efficientnet_b0',
    'val_accuracy': best,
}
with open('/content/classifier_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

!ls -lh /content/bird_classifier.onnx /content/bird_classifier.pth /content/classifier_meta.json
print(json.dumps(meta, indent=2)[:400])

In [ ]:
# sanity check: ONNX and PyTorch should agree before we ship it
!pip install -q onnxruntime
import onnxruntime as ort, numpy as np

session = ort.InferenceSession('/content/bird_classifier.onnx')
with torch.no_grad():
    torch_out = model(dummy).numpy()
onnx_out = session.run(None, {'input': dummy.numpy()})[0]
print('max difference:', float(np.abs(torch_out - onnx_out).max()))
assert np.allclose(torch_out, onnx_out, atol=1e-4), 'ONNX export does not match PyTorch'
print('ONNX matches PyTorch')

In [ ]:
from google.colab import files
files.download('/content/bird_classifier.onnx')
files.download('/content/bird_classifier.pth')
files.download('/content/classifier_meta.json')